# Animagine XL 4.0 - Kaggle GPU

1. ตั้งค่า Notebook ให้เปิด Internet
2. เลือก Accelerator เป็น GPU T4 x2 หรือ GPU ที่มี
3. รันเซลล์ติดตั้งเพียงครั้งเดียว
4. Restart session หาก Kaggle ขอให้ Restart
5. รันเซลล์โหลดโมเดลและสร้าง Gradio UI


In [ ]:
# Cell 1: ติดตั้งเวอร์ชันที่เข้ากันกันกับ Kaggle
%pip install -q --upgrade pip
%pip install -q 'diffusers==0.31.0' 'transformers==4.46.3' 'accelerate==1.1.1' 'huggingface_hub==0.26.2' 'gradio==4.44.1' 'gradio-client==1.4.3' 'safetensors>=0.4.5'
print('ติดตั้งเสร็จแล้ว: ถ้าหน้าต่างแจ้งให้ restart ให้เลือก Restart session')

In [ ]:
# Cell 2: โหลดโมเดลและเปิดหน้า UI
import os
import random
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '10'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU - ไปที่ Notebook Settings แล้วเลือก Accelerator เป็น GPU')

gpu_count = torch.cuda.device_count()
print(f'พบ GPU {gpu_count} ตัว: ' + ', '.join(torch.cuda.get_device_name(i) for i in range(gpu_count)))

model_id = 'cagliostrolab/animagine-xl-4.0'

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

load_kwargs = {
    'torch_dtype': torch.float16,
    'use_safetensors': True,
    'custom_pipeline': 'lpw_stable_diffusion_xl',
    'add_watermarker': False,
}
if hf_token:
    load_kwargs['token'] = hf_token

try:
    pipe = StableDiffusionXLPipeline.from_pretrained(model_id, **load_kwargs)
except Exception as exc:
    raise RuntimeError(
        'โหลด Hugging Face model ไม่สำเร็จ - ตรวจว่า Kaggle เปิด Internet, HF_TOKEN ถูกต้อง, '
        'และยอมรับเงื่อนไขที่ https://huggingface.co/cagliostrolab/animagine-xl-4.0 แล้ว'
    ) from exc

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
pipe.enable_model_cpu_offload(gpu_id=0)
print('โหลดโมเดลสำเร็จ พร้อมสร้างภาพ')

def generate_anime_v4(prompt, negative_prompt, aspect_ratio, steps, cfg_scale, seed):
    sizes = {
        'จัตุรัส (1:1)': (1024, 1024),
        'แนวตั้งมาตรฐาน (3:4)': (896, 1152),
        'แนวตั้ง 9:16': (832, 1216),
        'แนวนอน 16:9': (1216, 832),
    }
    width, height = sizes.get(aspect_ratio, (896, 1152))
    actual_seed = random.randint(0, 2147483647) if seed is None or int(seed) == -1 else int(seed)
    generator = torch.Generator(device='cpu').manual_seed(actual_seed)

    try:
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                num_inference_steps=int(steps),
                guidance_scale=float(cfg_scale),
                generator=generator,
            )
        torch.cuda.empty_cache()
        return result.images[0], actual_seed
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise gr.Error('VRAM ไม่พอ - ลดขนาดภาพหรือ Steps แล้วลองอีกครั้ง') from exc

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🌸 AI วาดภาพอนิเมะ - Animagine XL 4.0')
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label='Prompt', lines=3,
                value='1girl, silver hair, blue eyes, stylish casual clothes, sunset city background, masterpiece, high score, great score, absurdres',
            )
            neg_prompt_input = gr.Textbox(
                label='Negative Prompt', lines=3,
                value='lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score',
            )
            aspect_dropdown = gr.Dropdown(
                choices=['จัตุรัส (1:1)', 'แนวตั้งมาตรฐาน (3:4)', 'แนวตั้ง 9:16', 'แนวนอน 16:9'],
                value='แนวตั้งมาตรฐาน (3:4)', label='สัดส่วนภาพ',
            )
            with gr.Row():
                steps_slider = gr.Slider(15, 40, value=28, step=1, label='Steps')
                cfg_slider = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='CFG Scale')
            seed_input = gr.Number(value=-1, label='Seed (-1 = สุ่ม)', precision=0)
            generate_btn = gr.Button('วาดภาพ', variant='primary')
        with gr.Column():
            output_image = gr.Image(label='ผลลัพธ์')
            seed_output = gr.Number(label='Seed ที่ใช้', interactive=False)
    generate_btn.click(
        fn=generate_anime_v4,
        inputs=[prompt_input, neg_prompt_input, aspect_dropdown, steps_slider, cfg_slider, seed_input],
        outputs=[output_image, seed_output],
    )
demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)